In [ ]:
## S4: Feature Engineering
# Run order: 01_eda.ipynb -> 02_feature_engineering.ipynb -> 03_prepare_dataset.ipynb -> 04_extension_smote_xgboost.ipynb

from pathlib import Path
import joblib
import pandas as pd
import numpy as np

raw_path = Path('../data/paysim.csv')
engineered_path = Path('../data/engineered_features.pkl')
df = pd.read_csv(raw_path)
print(f'Loaded raw data: {df.shape}')

# [12]: Tao errorBalanceOrig
df['errorBalanceOrig'] = df['newbalanceOrig'] + df['amount'] - df['oldbalanceOrg']
print(df['errorBalanceOrig'].describe())
print(df.groupby('isFraud')['errorBalanceOrig'].mean())

# [13]: Tao errorBalanceDest
df['errorBalanceDest'] = df['oldbalanceDest'] + df['amount'] - df['newbalanceDest']
print(df.groupby('type')[['oldbalanceDest', 'newbalanceDest']].apply(lambda x: (x == 0).mean()))

# [14]: Tao balance_change
df['balance_change_orig'] = (df['newbalanceOrig'] - df['oldbalanceOrg']) / df['oldbalanceOrg'].replace(0, 1)
print((df['oldbalanceOrg'] == 0).sum())
print(df.groupby('isFraud')['balance_change_orig'].median())
df['balance_change_dest'] = (df['newbalanceDest'] - df['oldbalanceDest']) / df['oldbalanceDest'].replace(0, 1)

# [15]: Encode type
df = pd.get_dummies(df, columns=['type'], prefix='type')
print(df.filter(like='type_').head())

# [16]: Xu ly nameOrig, nameDest
df = df.drop(columns=['nameOrig', 'nameDest'])

# [17]: Kiem tra tong the
df.info()
print(df.head())
df = df.replace([np.inf, -np.inf], np.nan)
print(df.isnull().sum())

# [18]: Luu artifact trung gian cho buoc split
joblib.dump(df, engineered_path)
print(f'Saved engineered data to {engineered_path}')
